Evaluation and Framework Development for Indian Road Marking Detection Models Using Segmentation Techniques

Problem statement : This study focuses on exploring Indian road marking detection models
(mainly segmentation-based) and evaluating their performance.


• Milestones/Deliverables:
1. Explore and list down various publicly available Indian Road marking detection models.
2. Understand the evaluation criteria and dataset (prepare if needed).
3. Understand the working of detection model and if needed perform transfer learning to adapt the model for Indian road marking detection.
4. Create a python based framework to evaluate its performance [Images will be given for testing]
5. Prepare a ground-truth for a given images for benchmarking the performance.
6. Deliver final framework as open-source tool.

• Models: Students can use pre-trained segmentation models for framework development.

• Dataset : AU-Drone dataset.

The first step has been done to ensure seemless synchronization between colab and google drive

In [ ]:
from google.colab import drive
drive.flush_and_unmount()  # flush previous mount

# Now mount again
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!pip install segmentation_models_pytorch==0.3.2 --quiet
!pip install transformers accelerate --quiet

import os
import cv2
import numpy as np
from glob import glob
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

#Making sure all the paths are setup properly
import os

# Defining the structure based on provided paths
base_path = "/content/drive/MyDrive/augmented_dataset"
paths = {
    "base": base_path,
    "images": {
        "base": os.path.join(base_path, "images"),
        "train": os.path.join(base_path, "images/train"),
        "valid": os.path.join(base_path, "images/valid"),
        "test": os.path.join(base_path, "images/test")
    },
    "masks": {
        "base": os.path.join(base_path, "masks"),
        "train": os.path.join(base_path, "masks/train"),
        "valid": os.path.join(base_path, "masks/valid"),
        "test": os.path.join(base_path, "masks/test")
    }
}

paths


{'base': '/content/drive/MyDrive/augmented_dataset',
 'images': {'base': '/content/drive/MyDrive/augmented_dataset/images',
  'train': '/content/drive/MyDrive/augmented_dataset/images/train',
  'valid': '/content/drive/MyDrive/augmented_dataset/images/valid',
  'test': '/content/drive/MyDrive/augmented_dataset/images/test'},
 'masks': {'base': '/content/drive/MyDrive/augmented_dataset/masks',
  'train': '/content/drive/MyDrive/augmented_dataset/masks/train',
  'valid': '/content/drive/MyDrive/augmented_dataset/masks/valid',
  'test': '/content/drive/MyDrive/augmented_dataset/masks/test'}}

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_paths = sorted(glob(os.path.join(image_dir, '*.jpg')))
        self.mask_paths = sorted(glob(os.path.join(mask_dir, '*.png')))
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()



DataLoader Setup (AU-Drone)

In [ ]:
def get_transforms(phase):
    if phase == 'train':
        return A.Compose([
            A.Resize(512, 512),
            A.HorizontalFlip(),
            A.Normalize(),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(512, 512),
            A.Normalize(),
            ToTensorV2()
        ])


In [ ]:
batch_size = 4

train_dataset = SegmentationDataset(paths['images']['train'], paths['masks']['train'], transform=get_transforms('train'))
val_dataset = SegmentationDataset(paths['images']['valid'], paths['masks']['valid'], transform=get_transforms('valid'))
test_dataset = SegmentationDataset(paths['images']['test'], paths['masks']['test'], transform=get_transforms('valid'))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)


In [ ]:
def get_deeplab():
    return smp.DeepLabV3Plus(encoder_name="resnet50", in_channels=3, classes=1)


In [ ]:
def load_pretrained_weights(model, checkpoint_path):
    if checkpoint_path.endswith(".pth"):
        model.load_state_dict(torch.load(checkpoint_path))
    return model


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    global preds, masks
    for imgs, masks in loader:
        imgs, masks = imgs.cuda(), masks.cuda()
        preds = model(imgs)
        if isinstance(preds, dict): preds = preds['logits']
        preds = torch.sigmoid(preds).squeeze(1)
        loss = loss_fn(preds, masks.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.cuda(), masks.cuda()
            preds = model(imgs)
            if isinstance(preds, dict): preds = preds['logits']
            preds = (torch.sigmoid(preds).squeeze(1) > 0.5).float()
            intersection = (preds * masks).sum()
            union = preds.sum() + masks.sum()
            dice = 2.0 * intersection / (union + 1e-8)
            dice_scores.append(dice.item())
    return np.mean(dice_scores)


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = get_deeplab().to(device)  # change to get_deeplab() or get_segformer() accordingly
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.BCELoss()

for epoch in range(1, 11):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_dice = evaluate(model, val_loader)
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Dice={val_dice:.4f}")


NameError: name 'smp' is not defined

In [ ]:
# Step 1: Install and import
!pip install segmentation-models-pytorch albumentations -q

import os, cv2, torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score, jaccard_score, accuracy_score

# Step 2: Set paths
BASE_PATH = "/content/drive/MyDrive/augmented_dataset"
IMAGE_DIR = os.path.join(BASE_PATH, "images")
MASK_DIR = os.path.join(BASE_PATH, "masks")

# Step 3: Dataset class
class SegDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_fns = sorted(os.listdir(image_dir))
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_fns)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_fns[idx])
        mask_path = os.path.join(self.mask_dir, self.image_fns[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load mask and handle potential None value
        mask = cv2.imread(mask_path, 0)
        if mask is None:
            print(f"Warning: Mask not found or could not be loaded at {mask_path}. Skipping this sample.")
            return None  # or handle the error in a different way

        mask = (mask > 127).astype(np.uint8)
        mask = np.expand_dims(mask, axis=-1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]

        return image, mask.float()

# Step 4: Transforms
train_tfms = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(),
    A.Normalize(),
    ToTensorV2()
])

valid_tfms = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])

# Step 5: Loaders
train_ds = SegDataset(f"{IMAGE_DIR}/train", f"{MASK_DIR}/train", train_tfms)
valid_ds = SegDataset(f"{IMAGE_DIR}/valid", f"{MASK_DIR}/valid", valid_tfms)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=4, shuffle=False)

# Step 6: Transfer Learning Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = smp.DeepLabV3Plus(
    encoder_name='resnet101',         # pretrained encoder
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
).to(device)

# Optionally freeze encoder for transfer learning
for param in model.encoder.parameters():
    param.requires_grad = False

# Step 7: Training Setup
loss_fn = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Step 8: Metrics
def compute_metrics(preds, masks):
    preds = (preds > 0.5).float()
    preds_flat = preds.view(-1).cpu().numpy()
    masks_flat = masks.view(-1).cpu().numpy()

    return {
        "IoU": jaccard_score(masks_flat, preds_flat),
        "Dice": f1_score(masks_flat, preds_flat),
        "Acc": accuracy_score(masks_flat, preds_flat)
    }

# Step 9: Training Loop
for epoch in range(10):
    model.train()
    total_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = loss_fn(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"[Epoch {epoch+1}] Train Loss: {total_loss:.4f}")

    # Validation
    model.eval()
    with torch.no_grad():
        all_metrics = {"IoU": [], "Dice": [], "Acc": []}
        for imgs, masks in valid_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = torch.sigmoid(model(imgs))
            metrics = compute_metrics(preds, masks)
            for k in all_metrics:
                all_metrics[k].append(metrics[k])
        print(f"→ Val IoU: {np.mean(all_metrics['IoU']):.4f}, Dice: {np.mean(all_metrics['Dice']):.4f}, Acc: {np.mean(all_metrics['Acc']):.4f}")


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'NoneType'>